# All of Us long-read Phase 2 data

This notebook describes the AoU lrWGS Phase 2 cohort and builds a per-sample
covariate table (`covariates.v2.csv.gz`).

It is meant to run on the All of Us Researcher Workbench (needs `gsutil`,
`bcftools`, and `$WORKSPACE_BUCKET`).

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.ticker import MaxNLocator

pd.set_option("display.max_columns", None)

In [ ]:
def sh(cmd: str) -> None:
    print(cmd)
    get_ipython().system(cmd)


def gcs_cp(src: str, dest: str | None = None, *, force: bool = False) -> Path:
    dest_path = Path(dest) if dest else Path(src.rstrip("/").split("/")[-1])
    if dest_path.exists() and not force:
        print(f"already present: {dest_path}")
        return dest_path
    sh(f"gsutil cp {src} {dest_path}")
    return dest_path


def samples_from_gcs_vcf(gcs_path: str, out_txt: str) -> set[str]:
    sh(f"gsutil cat {gcs_path} | bcftools query -l > {out_txt}")
    return ids_from_txt(out_txt)


def ids_from_txt(path: str) -> set[str]:
    return {line.strip() for line in Path(path).read_text().splitlines() if line.strip()}

## Download inputs

Sample IDs from large VCFs are streamed with `gsutil cat` rather than copied
locally. Everything else is cached in the current directory.

In [ ]:
PHASE2_TSV = "lrWGS.phase2.sample.final_set.2025-05-27.tsv"
GLNEXUS_TSV = "12387.samples.used.in.GLnexus.tsv"
AGE_ZIP_EHR = "part.csv.gz"
PCA_FULL = "AoU_v9_full_training_pca.tsv"
RNA_MANIFEST = "bam_files_manifest_10k_rna_20250721.tsv"
PROTEOMICS_MANIFEST = "manifest_aou_proteomics_20250709.tsv"
QC_FLAGS_TSV = "production.v9.longreads.samples_flagged_by_qc.tsv"
EXTRACTION_TSV = "lr_dna_extraction_method_cdrv9.tsv"
CDR_V7_TXT = "sample-name-list.v7.txt"
CDR_V8_TXT = "sample-name-list.v8.txt"
CDR_V9_TXT = "sample-name-list.v9.txt"

if not Path(PHASE2_TSV).exists():
    sh(f"gsutil cp gs://prod-drc-broad/longreads/demo_group/{PHASE2_TSV} .")
    sh(f"gsutil cp {PHASE2_TSV} $WORKSPACE_BUCKET/scratch/kvg/")

if not Path(AGE_ZIP_EHR).exists():
    sh(
        "wget -O part.csv.gz "
        '"https://www.dropbox.com/scl/fi/8i5wt118f4gdgad0zy0pj/part.csv.gz'
        '?rlkey=8e3ic79337xyxe7hwoxghlzh2&st=xzmta5rz&dl=0"'
    )

if not Path(GLNEXUS_TSV).exists():
    sh(
        "wget -q -O 12387.samples.used.in.GLnexus.tsv "
        '"https://www.dropbox.com/scl/fi/xefq2l9m3jqgm2349v16f/12387.samples.used.in.GLnexus.tsv'
        '?rlkey=bx7t0kntmdnlldvbi31n61r1e&st=nuczxzmy&dl=0"'
    )

if not Path("chrM.g.vcf.bgz").exists():
    sh("gsutil cp gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/scratch/kvg/chrM.g.vcf.bgz* .")

gcs_cp(f"gs://prod-drc-broad/aou_pca/v9/Full/{PCA_FULL}")
gcs_cp("gs://prod-drc-broad/aou_pca/v9/Full/AoU_v9_full_training_pca_eigenvalues.tsv")

if not Path("AoU_v9_AFR_training_pca.tsv").exists():
    sh("gsutil cp gs://prod-drc-broad/aou_pca/v9/*/*pca.tsv .")

gcs_cp(f"gs://prod-drc-broad/aou_rnaseq/final_10k_rna_20250721/{RNA_MANIFEST}")
gcs_cp(f"gs://prod-drc-broad/aou_proteomics/final_10K_20250709/{PROTEOMICS_MANIFEST}")
# Always refresh CDR sample lists so a stale local copy cannot silently win.
gcs_cp(
    "gs://prod-drc-broad/alpha2-longreads-v1/longreads.alpha2.1040_research_ids.txt",
    CDR_V7_TXT,
    force=True,
)
gcs_cp(
    "gs://prod-drc-broad/v8/long-read-samples/vcfs/sample-name-list.txt",
    CDR_V8_TXT,
    force=True,
)
gcs_cp(
    "gs://prod-drc-broad/v9/lrWGS/vcfs/sample-name-list.txt",
    CDR_V9_TXT,
    force=True,
)
gcs_cp(f"gs://prod-drc-broad/longreads/releases/v9/{QC_FLAGS_TSV}")
gcs_cp(f"gs://prod-drc-broad/v9/lrWGS/extraction/{EXTRACTION_TSV}")

## Callset sample overlap

Compare sample IDs across the v7 joint callset, v8 echo callset, the
production joint-calling chr1 gVCF, and the integrated SV callset.

Samples in the SV callset but not the production gVCF are treated as
**withdrawn** and are added to the covariate table with `withdrawn=True`.

In [ ]:
samples_v7 = samples_from_gcs_vcf(
    "gs://prod-drc-broad/longreads/demo_group/phase1/Jointcall_dvcf.g.vcf.bgz.QualFT40.vcf.gz",
    "samples.v7.txt",
)
samples_v8 = samples_from_gcs_vcf(
    "gs://prod-drc-broad/v8/long-read-samples/vcfs/0000000000-echo_callset.vcf.gz",
    "samples.v8.txt",
)
good_samples = samples_from_gcs_vcf(
    "gs://fc-secure-839ae4b0-b566-4de0-b11b-c2da0982647e/production_joint_calling/outputs/Chromosomes/chr1.g.vcf.bgz",
    "good_samples.txt",
)
sv_samples = samples_from_gcs_vcf(
    "gs://prod-drc-broad/longreads/demo_group/integrated_sv_callset/v3_main.bcf",
    "sv_samples.txt",
)

withdrawn_samples = sv_samples - good_samples
all_callset_samples = sv_samples | samples_v8 | samples_v7

print(f"v7 joint call:              {len(samples_v7):>6}")
print(f"v8 echo callset:            {len(samples_v8):>6}")
print(f"production chr1 gVCF:       {len(good_samples):>6}")
print(f"SV v3:                      {len(sv_samples):>6}")
print(f"union(SV, v8, v7):          {len(all_callset_samples):>6}")
print(f"withdrawn (SV not in prod): {len(withdrawn_samples):>6}")

## Phase 2 sample set

In [ ]:
phase2 = pd.read_csv(PHASE2_TSV, sep="\t", dtype={"participant": str})
print(f"{len(phase2):,} rows, {phase2['participant'].nunique():,} unique participants")
phase2.head()

In [ ]:
is_ont = phase2["technology"].fillna("").str.contains("ONT")
is_pb = phase2["technology"].fillna("").str.contains("PacBio")

ont_participants = set(phase2.loc[is_ont, "participant"])
pacbio_participants = set(phase2.loc[is_pb, "participant"])

tech_summary = pd.Series(
    {
        "ONT": len(ont_participants),
        "PacBio": len(pacbio_participants),
        "ONT-only": len(ont_participants - pacbio_participants),
        "PacBio-only": len(pacbio_participants - ont_participants),
        "PacBio+ONT": len(ont_participants & pacbio_participants),
    },
    name="n_participants",
)
tech_summary

In [ ]:
glnexus = pd.read_csv(GLNEXUS_TSV, sep="\t", names=["participant", "center"], dtype=str)
missing_from_phase2 = sorted(set(glnexus["participant"]) - set(phase2["participant"]))
print(f"GLnexus samples: {len(glnexus):,}")
print(f"in GLnexus but not phase 2 table: {len(missing_from_phase2)}")

in_glnexus = phase2["participant"].isin(glnexus["participant"])
print("\nplatform counts among GLnexus samples present in the phase 2 table:")
print(phase2.loc[in_glnexus, "platform"].value_counts(dropna=False))

Participants are ordered PacBio-only → PacBio+ONT → ONT-only, then by combined
coverage. Each sequenced row is plotted on the technology track; coverage is
summed across a participant's rows.

In [ ]:
def tech_category(has_pacbio: bool, has_ont: bool) -> str:
    if has_pacbio and has_ont:
        return "PacBio+ONT"
    if has_pacbio:
        return "PacBio-only"
    if has_ont:
        return "ONT-only"
    return "Other"


cat_order = {"PacBio-only": 0, "PacBio+ONT": 1, "ONT-only": 2, "Other": 3}

phase2 = phase2.assign(is_ont=is_ont, is_pb=is_pb)
phase2["category"] = [
    tech_category(p in pacbio_participants, p in ont_participants)
    for p in phase2["participant"]
]
sorted_phase2 = (
    phase2.assign(category_order=phase2["category"].map(cat_order))
    .sort_values(["category_order", "coverage", "participant"])
    .drop(columns="category_order")
    .reset_index(drop=True)
)

participant_level = (
    phase2.assign(coverage=phase2["coverage"].fillna(0))
    .groupby("participant", as_index=False)
    .agg(
        combined_coverage=("coverage", "sum"),
        has_ONT=("is_ont", "any"),
        has_PacBio=("is_pb", "any"),
        n_rows=("participant", "count"),
    )
)
participant_level["category"] = [
    tech_category(pb, ont)
    for pb, ont in zip(participant_level["has_PacBio"], participant_level["has_ONT"])
]
participant_level = (
    participant_level.assign(category_order=participant_level["category"].map(cat_order))
    .sort_values(["category_order", "combined_coverage", "participant"])
    .drop(columns="category_order")
    .reset_index(drop=True)
)
participant_level.head()

In [ ]:
participants_sorted = sorted_phase2["participant"].unique()
x_map = {p: i for i, p in enumerate(participants_sorted)}

x_tech, y_tech, colors = [], [], []
for _, row in sorted_phase2.iterrows():
    tech = row["technology"]
    if not isinstance(tech, str):
        continue
    if "ONT" in tech:
        x_tech.append(x_map[row["participant"]])
        y_tech.append(1.0)
        colors.append("tab:blue")
    elif "PacBio" in tech:
        x_tech.append(x_map[row["participant"]])
        y_tech.append(2.0)
        colors.append("tab:red")

fig, (ax_cov, ax_tech) = plt.subplots(
    2, 1, figsize=(14, 8), sharex=True, gridspec_kw={"height_ratios": [4, 1]}
)
ax_cov.scatter(
    [x_map[p] for p in participant_level["participant"]],
    participant_level["combined_coverage"],
    c="black",
    alpha=0.7,
    s=10,
)
ax_cov.set_ylabel("Coverage (X)")
ax_cov.set_title("Coverage and technology by participant")

ax_tech.scatter(x_tech, y_tech, c=colors, alpha=0.7, s=12)
ax_tech.set_yticks([1, 2], ["ONT", "PacBio"])
ax_tech.set_xlabel("Participant index (PacBio-only → PacBio+ONT → ONT-only, then coverage)")
ax_tech.set_ylabel("Technology")
ax_tech.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))

plt.tight_layout()
plt.show()

## Covariate table

Index samples from the chrM gVCF, plus withdrawn samples that appear in the SV
callset but not the production joint-calling gVCF. Then attach TRGT,
PCA/population, Phase 2 metadata, RNA, proteomics, CDR version membership, QC
flags, age/ZIP3/EHR, and DNA extraction method.

In [ ]:
sh("bcftools query -l chrM.g.vcf.bgz > chrM.samples.txt")
sample_names = ids_from_txt("chrM.samples.txt")
n_chrM = len(sample_names)
sample_names |= withdrawn_samples
covariates = pd.DataFrame({"sample_id": sorted(sample_names)})
covariates["withdrawn"] = covariates["sample_id"].isin(withdrawn_samples)

trgt_vcfs = get_ipython().getoutput(
    "gsutil ls gs://fc-secure-8f7d6a20-04ce-40d7-8c88-aececeac3e09/CCS/*/outputs/GRCh38/TRGT/TR_Explorer_1.01/*.vcf.gz"
)
trgt_sample_names = {
    Path(path).name.split("_")[0] for path in trgt_vcfs if path.endswith(".vcf.gz")
}
covariates["has_trgt_calls"] = covariates["sample_id"].isin(trgt_sample_names)

n_controls = covariates["sample_id"].str.startswith(("HG", "NA")).sum()
n_added = len(sample_names) - n_chrM
print(f"{n_chrM:,} chrM samples + {n_added:,} withdrawn not in chrM = {len(covariates):,} rows")
print(f"{n_controls} HG/NA controls kept; {covariates['withdrawn'].sum():,} withdrawn")
print(f"{covariates['has_trgt_calls'].sum():,} have TRGT calls")
covariates.head()

In [ ]:
pca = pd.read_table(PCA_FULL, dtype={"s": str})
covariates = covariates.merge(pca, left_on="sample_id", right_on="s", how="left")

sample_to_pop = {}
for pca_path in sorted(Path(".").glob("AoU_v9_*_training_pca.tsv")):
    if pca_path.name == PCA_FULL:
        continue
    pop_code = pca_path.name.removeprefix("AoU_v9_").removesuffix("_training_pca.tsv")
    pop_ids = pd.read_table(pca_path, usecols=["s"], dtype={"s": str})["s"]
    sample_to_pop.update(dict.fromkeys(pop_ids, pop_code))

covariates["population"] = covariates["sample_id"].map(sample_to_pop)
print(covariates["population"].value_counts(dropna=False))

In [ ]:
sample_final_set = pd.read_table(PHASE2_TSV, dtype={"participant": str})
n_dupe_participants = sample_final_set["participant"].duplicated().sum()
print(f"{n_dupe_participants} extra rows from duplicated participants in the phase 2 table")

covariates = covariates.merge(
    sample_final_set,
    left_on="sample_id",
    right_on="participant",
    how="left",
    indicator=True,
)
print(covariates["_merge"].value_counts())

drop_from_phase2 = [
    "participant",
    "Read error rate (HG38)",
    "Supp.Aln. rate (HG38)",
    "RL Q3 - Q1",
    "Q30(%)",
    "Q20(%)",
    "Hap1 auN",
    "Hap2 auN",
    "Hap1 asm. len.",
    "Hap2 asm. len.",
    "basecall_config",
    "basecaller",
    "RL spread",
]
covariates = covariates.drop(columns=drop_from_phase2, errors="ignore")
print(f"rows={len(covariates):,}; unique sample_id={covariates['sample_id'].nunique():,}")

In [ ]:
both_present = covariates["population"].notna() & covariates["pred_ancestry"].notna()
match = (
    covariates.loc[both_present, "population"].str.upper()
    == covariates.loc[both_present, "pred_ancestry"].str.upper()
)
print("population vs pred_ancestry:")
print(match.value_counts().rename({True: "match", False: "mismatch"}))

ct = pd.crosstab(
    covariates.loc[both_present, "pred_ancestry"].str.upper(),
    covariates.loc[both_present, "population"],
)
plt.figure(figsize=(8, 6))
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues")
plt.xlabel("population (PCA training files)")
plt.ylabel("pred_ancestry (phase 2 table)")
plt.title("pred_ancestry vs population")
plt.tight_layout()
plt.show()

covariates = covariates.drop(columns=["pred_ancestry", "s", "_merge"], errors="ignore")

In [ ]:
plt.figure(figsize=(6, 5))
order = ["mid", "high"]
sns.boxplot(data=covariates, x="regime", y="coverage", order=order)
sns.stripplot(
    data=covariates, x="regime", y="coverage", order=order, color="black", alpha=0.3, size=2
)
plt.title("Coverage by regime")
plt.tight_layout()
plt.show()

In [ ]:
rna = pd.read_table(RNA_MANIFEST, dtype={"sampleid": str})
proteomics = pd.read_table(PROTEOMICS_MANIFEST, dtype={"research_id": str})
covariates["has_rna"] = covariates["sample_id"].isin(rna["sampleid"])
covariates["has_proteomics"] = covariates["sample_id"].isin(proteomics["research_id"])

print(f"has_rna:        {covariates['has_rna'].sum():>6} / {len(covariates)}")
print(f"has_proteomics: {covariates['has_proteomics'].sum():>6} / {len(covariates)}")

In [ ]:
qc_flags = pd.read_csv(QC_FLAGS_TSV, sep="\t", dtype={"participant": str})
n_qc = covariates["sample_id"].isin(qc_flags["participant"]).sum()
print(f"{n_qc} of {len(covariates)} samples flagged by QC")

age_zip_ehr = pd.read_csv(AGE_ZIP_EHR, dtype={"person_id": str})
n_before = len(covariates)
covariates = covariates.merge(
    age_zip_ehr, left_on="sample_id", right_on="person_id", how="left"
)
assert len(covariates) == n_before, "age/ZIP3/EHR merge changed row count"
covariates = covariates.drop(columns=["Unnamed: 0", "person_id"], errors="ignore")
print(f"{covariates['age_at_cdr'].isna().sum()} rows with no age/ZIP3/EHR match")

extraction = pd.read_table(EXTRACTION_TSV, dtype={"research_id": str})
covariates = covariates.merge(
    extraction[["research_id", "extraction_method"]],
    how="left",
    left_on="sample_id",
    right_on="research_id",
    indicator=True,
)
assert len(covariates) == n_before, "extraction merge changed row count"
covariates["extraction_method"] = covariates["extraction_method"].fillna("NA")
print(covariates["_merge"].value_counts())
covariates = covariates.drop(columns=["_merge", "research_id"], errors="ignore")

In [ ]:
n_controls = covariates["sample_id"].str.startswith(("HG", "NA")).sum()
print(f"keeping {n_controls} HG/NA controls")

covariates = covariates.rename(
    columns={
        "age_at_cdr": "age",
        "sample_id": "research_id",
        "sex": "inferred_sex",
    }
)
if "has_ehr_data" in covariates.columns:
    covariates["has_ehr_data"] = covariates["has_ehr_data"].fillna(0.0).astype(bool)

# Set CDR membership last from the three sample-name lists (never earlier),
# so merges cannot leave behind or overwrite these flags.
covariates = covariates.drop(
    columns=[c for c in covariates.columns if c.startswith("in_cdr_")],
    errors="ignore",
)
cdr_v7 = ids_from_txt(CDR_V7_TXT)
cdr_v8 = ids_from_txt(CDR_V8_TXT)
cdr_v9 = ids_from_txt(CDR_V9_TXT)
research_ids = covariates["research_id"].astype(str)
covariates["in_cdr_v7"] = research_ids.isin(cdr_v7)
covariates["in_cdr_v8"] = research_ids.isin(cdr_v8)
covariates["in_cdr_v9"] = research_ids.isin(cdr_v9)

print(f"{CDR_V7_TXT}: {len(cdr_v7):,} IDs → {covariates['in_cdr_v7'].sum():,} rows in table")
print(f"{CDR_V8_TXT}: {len(cdr_v8):,} IDs → {covariates['in_cdr_v8'].sum():,} rows in table")
print(f"{CDR_V9_TXT}: {len(cdr_v9):,} IDs → {covariates['in_cdr_v9'].sum():,} rows in table")
print(f"unique research_ids in v7 list but not in table: {len(cdr_v7 - set(research_ids)):,}")
print(f"unique research_ids in v8 list but not in table: {len(cdr_v8 - set(research_ids)):,}")
print(f"unique research_ids in v9 list but not in table: {len(cdr_v9 - set(research_ids)):,}")

final_cols = [
    "research_id",
    "biobank_id",
    "GC",
    "technology",
    "platform",
    "PacBioMethylationCaller",
    "has_trgt_calls",
    "has_rna",
    "has_proteomics",
    "has_ehr_data",
    "in_cdr_v7",
    "in_cdr_v8",
    "in_cdr_v9",
    "withdrawn",
    "zip3_as_string",
    "age",
    "inferred_sex",
    "sex_at_birth",
    "extraction_method",
    "population",
    "coverage",
    "is_AIAN",
    *[f"PC{i}" for i in range(1, 33)],
]
missing_cols = [c for c in final_cols if c not in covariates.columns]
if missing_cols:
    raise KeyError(f"missing expected columns: {missing_cols}")

covariates = covariates[final_cols]
covariates.head()

## Summaries

In [ ]:
print("CDR membership by platform (all rows)")
for col in ["in_cdr_v7", "in_cdr_v8", "in_cdr_v9"]:
    print(f"\n--- {col} ---")
    print(
        pd.crosstab(
            covariates["platform"].fillna("NA"),
            covariates[col],
            margins=True,
        )
    )

pacbio = covariates[covariates["technology"] == "PacBio"]
print(f"\nPacBio rows: {len(pacbio):,}\n")
for col in [
    "GC",
    "technology",
    "platform",
    "PacBioMethylationCaller",
    "has_trgt_calls",
    "has_rna",
    "has_proteomics",
    "has_ehr_data",
    "withdrawn",
    "inferred_sex",
    "sex_at_birth",
    "extraction_method",
    "population",
    "is_AIAN",
]:
    print(f"--- {col} (PacBio only) ---")
    print(pacbio[col].value_counts(dropna=False))
    print()

In [ ]:
n_controls = covariates["research_id"].str.startswith(("HG", "NA")).sum()
print(f"HG/NA controls: {n_controls:,} / {len(covariates):,}")
print(f"withdrawn: {covariates['withdrawn'].sum():,} / {len(covariates):,}")
missing_withdrawn = withdrawn_samples - set(covariates["research_id"])
print(f"withdrawn IDs not in final table: {len(missing_withdrawn)}")
if missing_withdrawn:
    print(sorted(missing_withdrawn))
print(f"final table: {len(covariates):,} rows, {covariates['research_id'].nunique():,} unique research_id")
covariates

## Export

In [ ]:
out = Path("covariates.v2.csv.gz")
covariates.to_csv(out, index=False, compression="gzip")
sh(f"gsutil cp {out} $WORKSPACE_BUCKET/scratch/kvg/")
print(f"wrote {out} ({out.stat().st_size / 1e6:.1f} MB)")